In [13]:
"""
# 2026-Y2-S1-MET-14: Predicting Credit Card Customer Churn
## Integrated Preprocessing Pipeline (group_pipeline.ipynb)
Dataset: BankChurners.csv (10,127 records, 23 original columns)

Individual Contributions:
1. IT25102085: Drop Irrelevant & Target Data Leakage Columns
2. IT25102090: Missing Value Imputation (Mode Imputation for 'Unknown' records)
3. IT25102091: Continuous Outlier Inspection & Signal Preservation (IQR)
4. IT25102092: Continuous Feature Correlation Analysis
5. IT25102086: Categorical Encoding (LabelEncoder fitted strictly on Train)
6. IT25102093: Stratified Train/Test Split & Feature Scaling (MinMaxScaler fitted strictly on Train)
"""
print("Group pipeline documentation loaded.")

Group pipeline documentation loaded.


In [14]:
import pandas as pd
import numpy as np

# 1. Load raw dataset
df = pd.read_csv('BankChurners.csv')
print(f"Original shape: {df.shape}")

# 2. Drop unique identifier and pre-computed Kaggle leakage columns
leakage_cols = [c for c in df.columns if c.startswith('Naive_Bayes_Classifier')]
drop_cols = ['CLIENTNUM'] + leakage_cols
df_cleaned = df.drop(columns=drop_cols)

print("Columns removed:", drop_cols)
print(f"Shape after removing leakage: {df_cleaned.shape}")

Original shape: (10127, 23)
Columns removed: ['CLIENTNUM', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']
Shape after removing leakage: (10127, 20)


In [15]:
# Identify and impute hidden 'Unknown' categories using column modes
missing_cols = ['Education_Level', 'Income_Category', 'Marital_Status']

for col in missing_cols:
    mode_val = df_cleaned[df_cleaned[col] != 'Unknown'][col].mode()[0]
    df_cleaned[col] = df_cleaned[col].replace('Unknown', mode_val)

print("Missing values successfully imputed with mode!")
print("Remaining 'Unknown' entries:", (df_cleaned[missing_cols] == 'Unknown').sum().sum())

Missing values successfully imputed with mode!
Remaining 'Unknown' entries: 0


In [16]:
# Check genuine continuous banking features using IQR boundaries
outlier_features = ['Credit_Limit', 'Total_Trans_Amt', 'Total_Trans_Ct']

Q1 = df_cleaned[outlier_features].quantile(0.25)
Q3 = df_cleaned[outlier_features].quantile(0.75)
IQR = Q3 - Q1

outlier_mask = ((df_cleaned[outlier_features] < (Q1 - 1.5 * IQR)) | (df_cleaned[outlier_features] > (Q3 + 1.5 * IQR)))
print("Outlier rows identified:", outlier_mask.any(axis=1).sum(), "out of", len(df_cleaned))
print("Action: Preserved all outliers to retain genuine customer churn behavior.")

Outlier rows identified: 1684 out of 10127
Action: Preserved all outliers to retain genuine customer churn behavior.


In [17]:
# Compute correlation matrix on continuous numeric features
numeric_df = df_cleaned.select_dtypes(include=['float64', 'int64'])
corr_matrix = numeric_df.corr().round(2)

unstacked = corr_matrix.unstack()
top_pairs = unstacked[unstacked < 1.0].sort_values(ascending=False).drop_duplicates()
print("Top correlated continuous feature pairs:")
print(top_pairs.head(3))

Top correlated continuous feature pairs:
Total_Trans_Amt        Total_Trans_Ct         0.81
Customer_Age           Months_on_book         0.79
Avg_Utilization_Ratio  Total_Revolving_Bal    0.62
dtype: float64


In [18]:
from sklearn.model_selection import train_test_split

# 1. Separate features from target
X = df_cleaned.drop(columns=['Attrition_Flag'])
y = (df_cleaned['Attrition_Flag'] == 'Attrited Customer').astype(int)

# 2. Stratified split MUST occur before encoding and scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data split complete! Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

Data split complete! Training samples: 8101, Testing samples: 2026


In [19]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# 1. Categorical Encoding (IT25102086) - Fit strictly on X_train
categorical_features = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']
for col in categorical_features:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

# 2. Numerical Feature Scaling (IT25102093) - Fit strictly on X_train using MinMaxScaler
features_to_scale = ['Credit_Limit', 'Total_Trans_Amt', 'Avg_Utilization_Ratio']
scaler = MinMaxScaler()
X_train[features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
X_test[features_to_scale] = scaler.transform(X_test[features_to_scale])

print("Pipeline transformations completed with ZERO data leakage!")
print("\nSample scaled training data (bounded 0 to 1):")
print(X_train[features_to_scale].head())

Pipeline transformations completed with ZERO data leakage!

Sample scaled training data (bounded 0 to 1):
      Credit_Limit  Total_Trans_Amt  Avg_Utilization_Ratio
2856      0.034213         0.070140               0.872362
6515      0.720658         0.100315               0.000000
7141      0.309323         0.214543               0.078392
632       0.050962         0.064243               0.512563
3496      1.000000         0.191469               0.034171


In [20]:
import os

# Create folders for final model-ready outputs
os.makedirs('data/processed', exist_ok=True)
os.makedirs('results/outputs', exist_ok=True)

# Combine features with target
train_processed = pd.concat([X_train, y_train], axis=1)
test_processed = pd.concat([X_test, y_test], axis=1)

# Export to both locations to satisfy course rubrics
train_processed.to_csv('data/processed/train_cleaned.csv', index=False)
test_processed.to_csv('data/processed/test_cleaned.csv', index=False)
train_processed.to_csv('results/outputs/train_cleaned.csv', index=False)
test_processed.to_csv('results/outputs/test_cleaned.csv', index=False)

print("Preprocessed files successfully exported to data/external/ and results/outputs/!")

Preprocessed files successfully exported to data/external/ and results/outputs/!
